In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install faiss-cpu #install FAISS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 52.2 MB/s eta 0:00:00:00:0100:01


In [3]:
import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating knowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [4]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
zs_res_150 = zs(prompt_150, candidate_labels=labels_150)
correct_index = zs_res_150['labels'].index(ans_150)
q1_ans = zs_res_150['scores'][correct_index]

print(f"Q1: {round(q1_ans, 3)}")

prompt_150_emb = model.encode([prompt_150], show_progress_bar=False)

# FAISS top 10
distances, indices = index.search(prompt_150_emb, k=10)
retrieved_indices = indices[0].tolist()

if 150 in retrieved_indices:
    rank_q2 = retrieved_indices.index(150) + 1
else:
    rank_q2 = "Not in top 10"

print(f"Retrieved FAISS Indices: {retrieved_indices}")
print(f"Q2: {rank_q2}")

Q1: 0.384
Retrieved FAISS Indices: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
Q2: 10


In [6]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [7]:
# mapping indices wrt their ce-scores
indexed_scores = list(zip(retrieved_indices, ce_scores))

# sort
reranked_docs = sorted(indexed_scores, key=lambda x: x[1], reverse=True)
reranked_indices = [item[0] for item in reranked_docs]

rank_150 = reranked_indices.index(150) + 1

print(f"Reranked Indices: {reranked_indices}")
print(f"Q3: {rank_150}")

Reranked Indices: [150, 847, 1693, 1906, 1269, 1532, 168, 576, 663, 1701]
Q3: 1


In [8]:
prompt_42 = str(train.iloc[42]['prompt'])

# FAISS top 5
prompt_42_emb = model.encode([prompt_42], show_progress_bar=False)
_, indices_42 = index.search(prompt_42_emb, k=5)
retrieved_docs_42 = [kb[i] for i in indices_42[0]]

# formatting
concatenated_docs = " ".join(retrieved_docs_42)
rag_string_42 = f"Context: {concatenated_docs} Question: {prompt_42}"

# token length calculation
tokenizer_bert = AutoTokenizer.from_pretrained('bert-base-uncased')
token_ids = tokenizer_bert.encode(rag_string_42, add_special_tokens=True, truncation=False)

print(f"Q4: {len(token_ids)}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q4: 216


In [9]:
true_doc_150 = kb[150]
rag_string_perfect = f"Context: {true_doc_150} Question: {prompt_150}"

res_q5 = zs(rag_string_perfect, candidate_labels=labels_150)
correct_idx_q5 = res_q5['labels'].index(ans_150)
q5_prob = res_q5['scores'][correct_idx_q5]

print(f"Q5: {round(q5_prob, 3)}")

unrelated_doc_999 = kb[999]
rag_string_adversarial = f"Context: {unrelated_doc_999} Question: {prompt_150}"

res_q6 = zs(rag_string_adversarial, candidate_labels=labels_150)
correct_idx_q6 = res_q6['labels'].index(ans_150)
q6_prob = res_q6['scores'][correct_idx_q6]

print(f"Q6: {round(q6_prob, 3)}")

Q5: 0.989
Q6: 0.529


In [10]:
hits = 0
total_rows = 100

for idx in range(total_rows):
    row = train.iloc[idx]
    prompt_text = str(row['prompt'])
    true_ans_text = str(row[row['answer']])
    
    # top 5 matches
    p_emb = model.encode([prompt_text], show_progress_bar=False)
    _, p_indices = index.search(p_emb, k=5)
    retrieved_chunks = [kb[i] for i in p_indices[0]]
    
    # check for ans in retrived chunks
    if any(true_ans_text in chunk for chunk in retrieved_chunks):
        hits += 1

hit_rate = (hits / total_rows) * 100
print(f"Q7: {round(hit_rate, 1)}%")

Q7: 73.0%


In [11]:
total_map = 0
rows_to_evaluate = 20
option_letters = ['A', 'B', 'C', 'D', 'E']

for idx in range(rows_to_evaluate):
    row = train.iloc[idx]
    prompt_text = str(row['prompt'])
    ground_truth = str(row['answer'])
    
    # top 5 using bi-enc
    p_emb = model.encode([prompt_text], show_progress_bar=False)
    _, p_indices = index.search(p_emb, k=5)
    retrieved_idx_list = p_indices[0].tolist()
    chunks_5 = [kb[i] for i in retrieved_idx_list]
    
    # rerank unsing cross-enc
    ce_pairs = [[prompt_text, chunk] for chunk in chunks_5]
    scores_ce = cross_encoder.predict(ce_pairs)
    best_chunk_idx = np.argmax(scores_ce)
    best_document = chunks_5[best_chunk_idx]
    
    # augment context
    rag_string = f"Context: {best_document} Question: {prompt_text}"
    
    # zero-shot prediction
    labels_list = [str(row[letter]) for letter in option_letters]
    res_inference = zs(rag_string, candidate_labels=labels_list)
    
    text_to_letter_map = {str(row[letter]): letter for letter in option_letters}
    
    # sort options
    predicted_letters_ranked = [text_to_letter_map[lbl] for lbl in res_inference['labels']]
    top_3_predictions = predicted_letters_ranked[:3]
    
    # MAP@3 score
    row_map = 0
    if top_3_predictions[0] == ground_truth:
        row_map = 1.0
    elif top_3_predictions[1] == ground_truth:
        row_map = 0.5
    elif top_3_predictions[2] == ground_truth:
        row_map = 1/3
        
    total_map += row_map

final_average_map = total_map / rows_to_evaluate
print(f"Q8: {round(final_average_map, 3)}")

Q8: 0.975
